# Chronic Kidney Disease (CKD) Prediction
**Logistic Regression baseline, Random Forest (scikit-learn) and Artificial Neural Network (TensorFlow/Keras)**

Pipeline: load -> clean -> EDA -> feature engineering -> imputation + preprocessing -> model training & tuning ->
cross-validation -> comparison -> robustness to missing data -> artifact export (used by the Flask app in `app.py`).

**Dataset:** UCI *Chronic Kidney Disease* (Rubini et al., 2015), 400 patients, 24 clinical fields + `classification`
(`ckd` / `notckd`). Download it from UCI or Kaggle ("Chronic Kidney Disease dataset") and save it as
`data/kidney_disease.csv`.

> **Not for clinical use.** This is an educational project on a small public dataset. Cleaning and feature
> engineering live in `src/features.py` and are shared with the Flask app.

In [ ]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import features as ft  # shared with the Flask app

SEED = 42
np.random.seed(SEED)
DATA_PATH = ROOT / "data" / "kidney_disease.csv"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Load data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Save the CKD csv as {DATA_PATH}")
raw = pd.read_csv(DATA_PATH)
raw.columns = raw.columns.str.strip()
print("Shape:", raw.shape)
raw.head()

In [ ]:
# The raw file is messy: stray tabs/spaces, '?' placeholders, numbers stored as text.
for col in ["dm", "cad", "classification"]:
    print(f"{col:15s}", raw[col].unique().tolist())
for col in ["pcv", "wc", "rc"]:
    print(f"{col:15s} dtype={raw[col].dtype}, sample={raw[col].astype(str).unique()[:6].tolist()}")

## 2. Cleaning
`ft.clean` strips whitespace/tabs, lower-cases text, turns `?`/blank into `NaN`, converts numeric columns and returns the
24 clinical columns. The target is encoded as `ckd = 1` (positive class) and `notckd = 0`.

In [ ]:
y = ft.encode_target(raw[ft.TARGET])
clean_df = ft.clean(raw)
print("Class counts:", y.map({1: "ckd", 0: "notckd"}).value_counts().to_dict())
print("Majority-class share:", round(max(y.mean(), 1 - y.mean()), 3))

missing_pct = clean_df.isna().mean().mul(100).sort_values(ascending=False)
print(f"Cells missing overall: {clean_df.isna().mean().mean():.1%}  |  rows with >=1 missing value: {clean_df.isna().any(axis=1).mean():.1%}")

## 3. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
y.map({1: "ckd", 0: "notckd"}).value_counts().plot.bar(ax=ax[0], rot=0, color=["#0f766e", "#94a3b8"], title="Class balance")
missing_pct.head(15).iloc[::-1].plot.barh(ax=ax[1], color="#e8890c", title="Top 15 columns by % missing")
plt.tight_layout(); plt.show()

In [ ]:
plot_df = clean_df.assign(Class=y.map({1: "ckd", 0: "notckd"}))
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, col in zip(axes, ["hemo", "sc", "bu", "pcv", "sod"]):
    sns.boxplot(data=plot_df, x="Class", y=col, ax=ax, palette=["#94a3b8", "#0f766e"], order=["notckd", "ckd"])
    ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 7))
for ax, col in zip(axes.ravel(), ["al", "sg", "su", "htn", "dm", "appet", "pe", "ane"]):
    sns.countplot(data=plot_df, x=col, hue="Class", ax=ax, hue_order=["notckd", "ckd"], palette=["#94a3b8", "#0f766e"])
    ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
num_corr = clean_df[ft.NUMERIC_RAW].assign(ckd=y).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(num_corr, cmap="vlag", center=0, annot=True, fmt=".1f", annot_kws={"size": 7})
plt.title("Correlation (numeric features + ckd target)")
plt.show()

## 4. Feature engineering
| Feature | Meaning |
|---|---|
| `UreaCreatinineRatio` | blood urea / serum creatinine |
| `LowHemoglobin` | hemoglobin < 12 g/dL (NaN if not measured) |
| `HighCreatinine` | serum creatinine > 1.2 mg/dL (NaN if not measured) |
| `Proteinuria` | urine albumin >= 1 (NaN if not measured) |
| `ComorbidityCount` | hypertension + diabetes + coronary artery disease |
| `SymptomCount` | pedal edema + anemia + poor appetite |
| `AbnormalUrinalysisCount` | abnormal RBC / pus cells + pus-cell clumps + bacteria |

Thresholds are illustrative reference values. Flags stay `NaN` when the underlying lab value is missing, so
imputation (below) never fabricates a measurement-based flag.

In [ ]:
df = ft.engineer(clean_df)
X = df[ft.MODEL_FEATURES]
print("Feature matrix:", X.shape)
eng_corr = X[ft.ENGINEERED_NUMERIC].assign(ckd=y).corr()["ckd"].drop("ckd").sort_values()
eng_corr.plot.barh(figsize=(7, 3.5), color="#0f766e", title="Engineered features: correlation with CKD")
plt.show()

## 5. Train/test split & preprocessing
Stratified 80/20 split. With only 400 rows, **a single 80-patient test set is noisy** - so sections 7-8 also report
5-fold cross-validation. The preprocessor (median / most-frequent **imputation** + scaling + one-hot) is fit on the
training set only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

preprocessor = ft.build_preprocessor()
Xtr = preprocessor.fit_transform(X_train)
Xte = preprocessor.transform(X_test)
feature_names = [n.split("__", 1)[1] for n in preprocessor.get_feature_names_out()]
print(f"Train: {Xtr.shape} | Test: {Xte.shape} | features after encoding: {len(feature_names)}")

In [ ]:
results, probas = {}, {}

def evaluate(name, proba, threshold=0.5, verbose=True):
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred),             # sensitivity: share of CKD patients caught
        "Specificity": tn / (tn + fp),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    }
    probas[name] = proba
    if verbose:
        print(f"=== {name} (threshold={threshold}) | missed CKD cases (false negatives): {fn} ===")
        print(classification_report(y_test, pred, target_names=["notckd", "ckd"], digits=3))
    return pred

baseline = DummyClassifier(strategy="most_frequent").fit(Xtr, y_train)
print("Majority-class baseline accuracy:", round(accuracy_score(y_test, baseline.predict(Xte)), 4))

## 6. Models

### 6.1 Logistic Regression (baseline)

In [ ]:
lr = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED).fit(Xtr, y_train)
_ = evaluate("Logistic Regression", lr.predict_proba(Xte)[:, 1])

### 6.2 Random Forest (tuned with randomized search, 5-fold CV on the training set)

In [ ]:
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 8, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.5],
}
search = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=1), param_distributions=param_dist, n_iter=20,
    scoring="f1", cv=StratifiedKFold(5, shuffle=True, random_state=SEED), random_state=SEED, n_jobs=-1, verbose=1,
)
search.fit(Xtr, y_train)
rf = search.best_estimator_
print("Best CV F1:", round(search.best_score_, 4))
print("Best params:", search.best_params_)
_ = evaluate("Random Forest", rf.predict_proba(Xte)[:, 1])

In [ ]:
imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values().tail(12)
perm = permutation_importance(rf, Xte, y_test, n_repeats=20, random_state=SEED, scoring="f1", n_jobs=-1)
perm = pd.Series(perm.importances_mean, index=feature_names).sort_values().tail(12)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
imp.plot.barh(ax=ax[0], color="#0f766e", title="Random Forest impurity importance (top 12)")
perm.plot.barh(ax=ax[1], color="#e8890c", title="Permutation importance on test set (top 12)")
plt.tight_layout(); plt.show()

### 6.3 Artificial Neural Network (TensorFlow / Keras)
`Dense(32, relu) -> Dropout(0.3) -> Dense(16, relu) -> Dropout(0.2) -> Dense(1, sigmoid)`, Adam, binary cross-entropy,
early stopping on validation loss. A small network with dropout is used because there are only ~320 training rows.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)

ann = keras.Sequential([
    layers.Input(shape=(Xtr.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
ann.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
            metrics=["accuracy", keras.metrics.AUC(name="auc")])
ann.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5),
]
history = ann.fit(Xtr.astype("float32"), y_train.values, validation_split=0.2,
                  epochs=200, batch_size=16, callbacks=callbacks, verbose=2)
ann.save(MODEL_DIR / "ann_model.keras")

In [ ]:
hist = pd.DataFrame(history.history)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
hist[["loss", "val_loss"]].plot(ax=ax[0], title="Loss")
hist[["accuracy", "val_accuracy"]].plot(ax=ax[1], title="Accuracy")
plt.tight_layout(); plt.show()

In [ ]:
ann_proba = ann.predict(Xte.astype("float32"), verbose=0).ravel()
_ = evaluate("ANN", ann_proba)

## 7. Model comparison (held-out test set)
In a screening setting a **missed CKD case (false negative)** is costlier than a false alarm, so read *Recall* and
*Specificity* next to accuracy.

In [ ]:
comparison = pd.DataFrame(results).T.round(4).sort_values("ROC-AUC", ascending=False)
display(comparison)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for name, p in probas.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    ax[0].plot(fpr, tpr, label=f"{name} (AUC={results[name]['ROC-AUC']:.3f})")
ax[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
ax[0].set(xlabel="False positive rate", ylabel="True positive rate (recall)", title="ROC curves")
ax[0].legend(loc="lower right")
comparison[["Accuracy", "Precision", "Recall", "Specificity", "F1"]].plot.bar(ax=ax[1], rot=15, title="Metrics at threshold 0.5")
ax[1].set_ylim(0.5, 1.02)
plt.tight_layout(); plt.show()

In [ ]:
final = {k: v for k, v in probas.items() if k in ("Logistic Regression", "Random Forest", "ANN")}
fig, axes = plt.subplots(1, len(final), figsize=(5 * len(final), 4.2))
for ax, (name, p) in zip(np.atleast_1d(axes), final.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, (p >= 0.5).astype(int), display_labels=["notckd", "ckd"],
                                            ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.tight_layout(); plt.show()

## 8. Cross-validation (more reliable than one 80-row test set)
Each fold re-fits the **whole pipeline** (imputer, scaler, encoder, model) on that fold's training part, so there is no
leakage from the validation fold.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv_summary = {}
for name, model in {"Logistic Regression": lr, "Random Forest": rf}.items():
    pipe = make_pipeline(ft.build_preprocessor(), clone(model))
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_summary[name] = {m: f"{scores[f'test_{m}'].mean():.3f} +/- {scores[f'test_{m}'].std():.3f}" for m in scoring}
pd.DataFrame(cv_summary).T

## 9. Robustness: what if lab values are missing?
In practice not every test is ordered. We randomly blank out a fraction of the **test** cells and let the fitted
pipeline impute them (averaged over 5 random masks). A steep drop means the model leans on a few lab values.

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for frac in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    for name, model in {"Logistic Regression": lr, "Random Forest": rf}.items():
        accs, recs = [], []
        for _ in range(1 if frac == 0 else 5):
            Xm = X_test.mask(rng.random(X_test.shape) < frac)
            pred = model.predict(preprocessor.transform(Xm))
            accs.append(accuracy_score(y_test, pred)); recs.append(recall_score(y_test, pred))
        rows.append({"missing_fraction": frac, "model": name, "accuracy": np.mean(accs), "recall": np.mean(recs)})
robust = pd.DataFrame(rows)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for metric, a in zip(["accuracy", "recall"], ax):
    sns.lineplot(data=robust, x="missing_fraction", y=metric, hue="model", marker="o", ax=a)
    a.set_title(f"Test {metric} vs. fraction of values blanked")
plt.tight_layout(); plt.show()

## 10. Export artifacts for the Flask app
`preprocessor.joblib`, `rf_model.joblib`, `ann_model.keras` (saved above) and `metrics.json` go to `models/`.

In [ ]:
joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")
joblib.dump(rf, MODEL_DIR / "rf_model.joblib", compress=3)

metrics = {
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "majority_baseline_accuracy": round(float(accuracy_score(y_test, baseline.predict(Xte))), 4),
    "rf_best_params": search.best_params_,
    "cv_5fold_train": cv_summary,
    "test_metrics": {k: {m: round(float(v), 4) for m, v in r.items()}
                     for k, r in results.items() if k in ("Random Forest", "ANN", "Logistic Regression")},
}
(MODEL_DIR / "metrics.json").write_text(
    json.dumps(metrics, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print(json.dumps(metrics["test_metrics"], indent=2))
print("Saved:", sorted(p.name for p in MODEL_DIR.iterdir() if p.name != ".gitkeep"))

### Sanity check - run the exact code path the Flask app uses (including a patient with missing labs)

In [ ]:
from predictor import CKDPredictor, validate_patient

predictor = CKDPredictor(MODEL_DIR)
print("Loaded models:", predictor.available_models)

example = {"age": 62, "bp": 90, "sg": 1.010, "al": 3, "su": 1, "rbc": "abnormal", "pc": "abnormal", "pcc": "present",
           "ba": "notpresent", "bgr": 180, "bu": 80, "sc": 4.5, "sod": 128, "pot": 5.6, "hemo": 8.5, "pcv": 26,
           "wc": 9800, "rc": 3.1, "htn": "yes", "dm": "yes", "cad": "no", "appet": "poor", "pe": "yes", "ane": "yes"}
for label, payload in [("full record", example),
                       ("only 8 fields", {k: example[k] for k in ["age", "bp", "al", "sc", "hemo", "htn", "dm", "appet"]})]:
    _, record, provided = validate_patient(payload)
    out = predictor.predict([record])[0]
    print(f"{label:14s} ({provided} fields) -> p(CKD)={out['ckd_probability']:.3f}  risk={out['risk_level']}")

## 11. Summary & limitations
* Three models were trained on identical, leakage-free preprocessing and compared on a held-out set **and** with 5-fold CV.
* Published results on this dataset are often in the high 90s because several lab values (hemoglobin, creatinine,
  specific gravity, albumin) separate the classes very strongly - do not read that as clinical-grade performance.
* Limitations: only 400 patients from a single source, roughly a tenth of all cells missing, no sex variable, no external validation set.
  A real system would need prospective validation, calibration and clinician oversight.

**Possible next steps:** probability calibration, SHAP explanations, gradient boosting, external validation on another cohort.